# Image Processing Basics Lab

This notebook reorganizes the `#3 영상 처리 기초` lecture into a compact hands-on lab.

Topics:
- thresholding and Otsu binarization
- morphology
- gamma correction and histogram equalization
- filtering
- geometric transforms

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = Path("lab-session/02-image-processing-basics")

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

DOG_PATH = DATA_DIR / "dog1.jpg"
VIEW_PATH = DATA_DIR / "view.png"
CAT_PATH = DATA_DIR / "cat.jpg"
print("Base dir:", BASE_DIR.resolve())

## Thresholding and Otsu binarization

In [ ]:
gray = cv2.imread(str(DOG_PATH), cv2.IMREAD_GRAYSCALE)
if gray is None:
    raise FileNotFoundError(DOG_PATH)

_, binary_fixed = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
otsu_threshold, binary_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title("Gray")
axes[1].imshow(binary_fixed, cmap="gray")
axes[1].set_title("Fixed threshold")
axes[2].imshow(binary_otsu, cmap="gray")
axes[2].set_title(f"Otsu ({otsu_threshold:.1f})")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## Morphology

We use the threshold result as input and compare erosion, dilation, opening, and closing.

In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
erosion = cv2.erode(binary_otsu, kernel, iterations=1)
dilation = cv2.dilate(binary_otsu, kernel, iterations=1)
opening = cv2.morphologyEx(binary_otsu, cv2.MORPH_OPEN, kernel)
closing = cv2.morphologyEx(binary_otsu, cv2.MORPH_CLOSE, kernel)

images = [binary_otsu, erosion, dilation, opening, closing]
titles = ["Binary", "Erosion", "Dilation", "Opening", "Closing"]

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, image, title in zip(axes, images, titles):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

## Gamma correction and histogram equalization

In [ ]:
def gamma_correction(image, gamma):
    image_float = image.astype(np.float32) / 255.0
    corrected = np.power(image_float, 1.0 / gamma)
    return np.uint8(np.clip(corrected * 255.0, 0, 255))

gamma_dark = gamma_correction(gray, 0.5)
gamma_bright = gamma_correction(gray, 2.0)
equalized = cv2.equalizeHist(gray)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
tone_images = [gray, gamma_dark, gamma_bright, equalized]
tone_titles = ["Original", "Gamma 0.5", "Gamma 2.0", "Equalized"]
for idx, (image, title) in enumerate(zip(tone_images, tone_titles)):
    axes[0, idx].imshow(image, cmap="gray")
    axes[0, idx].set_title(title)
    axes[0, idx].axis("off")
    axes[1, idx].hist(image.ravel(), bins=32, color="black")
    axes[1, idx].set_title(f"Hist: {title}")
plt.tight_layout()

## Filtering

The lecture introduces convolution-based filtering, including blur and emboss examples.

In [ ]:
view_gray = cv2.imread(str(VIEW_PATH), cv2.IMREAD_GRAYSCALE)
if view_gray is None:
    raise FileNotFoundError(VIEW_PATH)

average_kernel = np.ones((5, 5), dtype=np.float32) / 25.0
average_blur = cv2.filter2D(view_gray, -1, average_kernel)
gaussian_blur = cv2.GaussianBlur(view_gray, (5, 5), 0)
emboss_kernel = np.array([[-1.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
emboss = cv2.filter2D(view_gray.astype(np.int16), -1, emboss_kernel)
emboss = np.uint8(np.clip(emboss + 128, 0, 255))

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
filter_images = [view_gray, average_blur, gaussian_blur, emboss]
filter_titles = ["Original", "Average blur", "Gaussian blur", "Emboss"]
for ax, image, title in zip(axes, filter_images, filter_titles):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

## Geometric transforms

In [ ]:
cat_bgr = cv2.imread(str(CAT_PATH))
if cat_bgr is None:
    raise FileNotFoundError(CAT_PATH)

h, w = cat_bgr.shape[:2]
translated = cv2.warpAffine(cat_bgr, np.float32([[1, 0, 60], [0, 1, 40]]), (w, h))
rotated = cv2.warpAffine(cat_bgr, cv2.getRotationMatrix2D((w / 2, h / 2), 25, 1.0), (w, h))
reflected = cv2.flip(cat_bgr, 1)
scaled = cv2.resize(cat_bgr, dsize=(0, 0), fx=1.2, fy=1.2, interpolation=cv2.INTER_LINEAR)
scaled = scaled[:h, :w]
if scaled.shape[:2] != (h, w):
    padded = np.zeros_like(cat_bgr)
    padded[: scaled.shape[0], : scaled.shape[1]] = scaled
    scaled = padded

geo_images = [cat_bgr, translated, rotated, reflected]
geo_titles = ["Original", "Translate", "Rotate", "Reflect"]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, image, title in zip(axes, geo_images, geo_titles):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

summary = np.vstack([
    np.hstack([cv2.resize(cat_bgr, (320, 240)), cv2.resize(translated, (320, 240))]),
    np.hstack([cv2.resize(rotated, (320, 240)), cv2.resize(reflected, (320, 240))]),
])
summary_path = OUTPUT_DIR / "image_processing_summary.jpg"
cv2.imwrite(str(summary_path), summary)
print("Saved summary image to:", summary_path.resolve())

## Optional script-based extensions

Run these from the same folder when you want the full OpenCV window-based demos:
- `python examples/01_threshold_otsu.py`
- `python examples/02_morphology.py`
- `python examples/03_gamma_histogram.py`
- `python examples/04_filtering.py`
- `python examples/05_geometric_transform.py`
- `python examples/06_dissolve_transition.py`